In [1]:
import os
import sys 
import numpy as np
sys.path.append('..')
# sys.path.append('../..')
os.getcwd()

from dotenv import load_dotenv
load_dotenv(override=True)


True

In [2]:
import src 

sys.modules['chatgenie'] = src
import src as cg

from src.rag.extr import ExTrRAG

ImportError: sklearn not installed , Please install scikit-learn


In [3]:
openai = cg.llm.LLM(llm_type='openai', api_key=os.getenv('OPENAI_API_KEY'))
embeder = cg.embedder.Embedder(embedder_type='openai', api_key=os.getenv('OPENAI_API_KEY'), model='text-davinci-003', dimesion=os.getenv('MONGO_DB_DIMENSION'))
vectordb = cg.vectordb.VectorDB(
                db_type="milvus",
                collection_name=os.getenv('MILVUS_COLLECTION'),
                dimensions=1536,  # Set explicit dimension value
             
            )

/users/oshan/Dev/financial-document-based-agent-system/src/DocHandler/notebooks/../src/llm/interface.py:18: UserWarning: Parameters {'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  self._cls_concrete = OpenAILlm(config)
/users/oshan/Dev/financial-document-based-agent-system/.venv/lib/python3.12/site-packages/langchain_openai/embeddings/base.py:313: UserWarning: WARNING! encoding_format is not default parameter.
                    encoding_format was transferred to model_kwargs.
                    Please confirm that encoding_format is what you intended.
  warnings.warn(


Collection 'finacial_docuent_based_agent_system' already exists. Skipping creation.


## ExTr RAG 

In [4]:
extr_rag = ExTrRAG(
    llm=openai,
    embedder=embeder,
    db=vectordb,
    memory="none",
    history=True
)

/users/oshan/Dev/financial-document-based-agent-system/src/DocHandler/notebooks/../src/rag/extr.py:55: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  self.context = ConversationBufferWindowMemory(k=5) # XXX: memory vs. context size


In [5]:
# extr_rag.add_document("Elle (sport) - Wikipedia.pdf")

In [6]:
# extr_rag.chat("How to play elle?")

In [7]:
# extr_rag.add_document(source="../data/Easy_recipes.pdf")

In [8]:
# extr_rag.chat("Give me a set of dishes that I can create with chicken and rice?")

## Document Loading test


In [9]:
extr_rag.loader.dry_run = False

In [10]:
pdf_path = "/users/oshan/Dev/financial-document-based-agent-system/data/wipo_pub_rn2021_18e.pdf"

In [14]:
print(f"Testing Processing for: {os.path.basename(pdf_path)} ---")

Testing Processing for: wipo_pub_rn2021_18e.pdf ---


In [ ]:
from src.loaders.dockling_loader import DocklingLoader
from src.chunkers.mdx import MdxChunker

loader = DocklingLoader(server_url="http://localhost:8080/documents/convert")
# creates the .md file locally
raw_data = loader.load_data(pdf_path) 
print(f"Conversion complete. Markdown size: {len(raw_data['data'][0]['content'])} chars")

chunker = MdxChunker() 
chunks = chunker.create_chunks(loader, pdf_path)

print(f"\n HUNKING RESULTS:")
print(f"Total Chunks Created: {len(chunks['ids'])}")


Conversion complete. Markdown size: 270443 chars

 HUNKING RESULTS:
Total Chunks Created: 381


In [9]:
for i in range(min(30, len(chunks['ids']))):
    chunk_id = chunks['ids'][i]
    content = chunks['documents'][i]
    meta = chunks['metadatas'][i]
    
    print(f"\n CHUNK #{i} [ID: {chunk_id}]")
    print(f"   Metadata: {meta}")
    print(f"   {content.strip()[:300]}...") 




 CHUNK #0 [ID: 63fcbc8f024ec63f49c4d469eb2485faa09276d36ec30d71e0c22b02e062f628_0]
   Metadata: {'source': '/users/oshan/Dev/financial-document-based-agent-system/data/wipo_pub_rn2021_18e.pdf', 'processed_path': '/users/oshan/Dev/financial-document-based-agent-system/data/wipo_pub_rn2021_18e.md', 'parser': 'dockling_remote'}
   ## Annual financial report and financial statements

Year to December 31, 2020

picture-1.png

## World Intellectual Property Organization

Annual Financial Report and Financial Statements

Year to December 31, 2020

## CONTENTS...

 CHUNK #1 [ID: 63fcbc8f024ec63f49c4d469eb2485faa09276d36ec30d71e0c22b02e062f628_1]
   Metadata: {'source': '/users/oshan/Dev/financial-document-based-agent-system/data/wipo_pub_rn2021_18e.pdf', 'processed_path': '/users/oshan/Dev/financial-document-based-agent-system/data/wipo_pub_rn2021_18e.md', 'parser': 'dockling_remote'}
   | ANNUAL FINANCIAL REPORT ................................................................................

### Questions generation and saving in the VDB

In [ ]:
records = extr_rag.loader.extr_load(pdf_path)


if len(records) > 0:
    first_record = records[0]
    
    # Structure & IDs
    print(f"Chunk ID: {first_record.get('id', 'MISSING')}")
    
    # The Chunk Content
    content_preview = first_record.get('documents', '')[:150].replace('\n', ' ')
    print(f"\n Content Preview:\n'{content_preview}...'")
    
    # The Questions
    questions = first_record.get('questions', [])
    print(f"\n Generated Questions ({len(questions)}):")
    for q in questions:
        print(f"   - {q}")
        
    # The Embeddings
    embedding = first_record.get('embeddings', [])
    emb_array = np.array(embedding)
    print(f"\n Embedding Shape: {emb_array.shape} (First 3 val: {emb_array[:3]})")

    # Milvus Compatibility
    required_keys = ['_id', 'content', 'text_embedding', 'meta_data']
    missing_keys = [key for key in required_keys if key not in first_record]
    if missing_keys:
        print(f"\n WARNING: Missing keys for Milvus Schema: {missing_keys}")
    else:
        print("\n SCHEMA CHECK: All keys ready for Milvus!")

else:
    print(" Error: No records were returned. Check Loader/Chunker.")

/users/oshan/Dev/financial-document-based-agent-system/.venv/lib/python3.12/site-packages/langchain_openai/chat_models/base.py:2067: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(
ic| question_response: Questions(questions=['What is the title of the document?', 'What is the year covered in the financial report?', 'What organization is the financial report from?'])
ic| questions: ['What is the title of the document?',
                'What is the year covered in the financial report?',
                'What organization is the financial report from?']
ic| question_response: Questions(questions=['What is the topic of the annual financial report?', 'What is discussed in the financial stateme

TypeError: object of type 'NoneType' has no len()